# Step 4 — Model Inference & GenAI Decision Explanations

This step shifts from ML ops to LLM ops. We take the model's predictions and feature importances, and use Generative AI to translate these raw probabilities into human-readable, persona-specific explanations.

- **`predict_batch`**: Scores structured rows using our loaded RandomForest artifact.
- **`top_feature_names`**: Surfaces local/global feature importances.
- **`explain_personas`**: Uses the LLM to generate targeted textual reasoning. For instance, explaining the exact same decision differently to a 'Customer' (empathetic, non-technical) versus a 'Claims Adjuster' (data-driven, fraud-focused).


In [1]:
from pathlib import Path
import sys

_CWD = Path.cwd().resolve()
PROJECT_ROOT = _CWD.parent if _CWD.name == "notebooks" else _CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = C:\coding\boltech


In [2]:
import joblib
from app.config import get_settings

bundle_path = get_settings().artifacts_dir / "approval_model.joblib"
bundle = joblib.load(bundle_path)
sorted(bundle.keys())


['metadata', 'pipeline']

In [3]:
import pandas as pd
from app.ml.dataset import load_claims_training_frame
from app.ml.inference_inputs import dataframe_row_to_claim
from app.ml.predict import predict_batch, top_feature_names

data_path = PROJECT_ROOT / "claim_use_case_dataset.xlsx"
X, y = load_claims_training_frame(data_path)
preds, probs = predict_batch(X.iloc[:8])
pd.DataFrame({"approved_hat": preds, "p_approve": probs})


c:\coding\boltech\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\coding\boltech\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,approved_hat,p_approve
0,1,0.82
1,1,1.00
2,1,0.81
3,1,0.94
4,1,0.87
5,1,0.92
6,1,0.98
7,1,0.99


In [4]:
top_feature_names(14)


[('num__issue_desc_len', 0.1491829374980272),
 ('num__fee_to_rrp', 0.047688316248773344),
 ('num__purchase_to_end_days', 0.04253758082105456),
 ('num__rrp', 0.036311286703365475),
 ('num__balanceRRP', 0.03502487300200859),
 ('num__symptom_count', 0.03484535081597954),
 ('num__balanceRRP_log1p', 0.03465292218301973),
 ('num__rrp_log1p', 0.03413123924976822),
 ('num__oldBalanceRRP_log1p', 0.03360264017886209),
 ('num__oldBalanceRRP', 0.03192848023211954),
 ('num__product_name_len', 0.02394860167890343),
 ('num__policy_length_days', 0.022643332173731358),
 ('num__touchScreen', 0.0222439107978053),
 ('num__excessFee_log1p', 0.018103428604759008)]

### Multi-persona explanations

Requires artifacts. **`GEMINI_API_KEY`** enables live Gemini; stubs otherwise.

- **`explain_personas` is `async`** so the notebook can use top-level **`await`** in IPython (same event loop; avoid `asyncio.run()` here).
- **Latency** is mostly **remote LLM time**. With two personas, **`asyncio.gather`** issues two calls **in parallel** when **`GEMINI_RPM`** is **unset**; with **`GEMINI_RPM`** set, spacing still applies between *start* times (see `app/genai/gemini.py`).


In [ ]:
import textwrap
import time

# explain_personas is async: Jupyter/IPython supports top-level await on the running event loop.
from app.genai.service import explain_personas
from app.ml.predict import predict_batch, top_feature_names
from app.schemas import Persona

claim = dataframe_row_to_claim(X, 0)
small = X.iloc[[0]]
preds, probs = predict_batch(small)
pa = float(probs[0])
if pa >= 0.75:
    band_txt = "likely_approve"
elif pa >= 0.55:
    band_txt = "lean_approve"
elif pa >= 0.45:
    band_txt = "borderline"
elif pa >= 0.25:
    band_txt = "lean_decline"
else:
    band_txt = "likely_decline"

payload = {
    "approved": bool(int(preds[0]) == 1),
    "probability_approved": pa,
    "risk_band": band_txt,
}
feats = top_feature_names(10)

_wrap = 100
_t0 = time.perf_counter()
outs = await explain_personas(
    claim=claim,
    personas=[Persona.customer, Persona.claims_adjuster],
    model_payload=payload,
    top_features=feats,
)
elapsed = time.perf_counter() - _t0

for role, text in outs.items():
    title = role.replace("_", " ").title()
    bar = "=" * min(72, _wrap)
    print(f"\n{bar}\n {title}\n{bar}")
    body = text.strip() or "(empty)"
    for block in body.split("\n\n"):
        print(textwrap.fill(block.strip(), width=_wrap, replace_whitespace=False))
        print()
print(f"\n-- {len(outs)} persona(s) · {elapsed:.1f}s wall time --\n")


c:\coding\boltech\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\coding\boltech\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


=== customer ===
{
  "customer": "We have reviewed your theft claim for your WUAWEI-AAA1 smartphone, and we are pleased to inform you that your claim has been approved. \n\n### Why your claim was approved\n* **Policy Coverage:** Your claim falls within the terms of your SEADLDTHEFT12 policy, which specifically covers theft.\n* **Claim Details:** The information provided regarding the circumstances of the incident was consistent with the coverage requirements for your active policy.\n* **Account Status:** Your policy was active and in good standing at the time of the reported incident.\n\n### Next Steps\n* **Excess Fee:** Please be prepared to pay the excess fee of 1989.0 SEK to finalize the processing of your claim.\n* **Documentation:** Keep an eye on your email for further instructions from our claims team regarding the next steps for your device replacement.\n* **Support:** If you have any questions o
=== claims_adjuster ===
{
  "claims_adjuster": "### Decision Summary\n* **Status:*

*(Explanations are printed in the cell above; remove `GEMINI_RPM` or lower personas count if you want faster parallel calls.)*
